In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

sns.set_context("poster")
sns.set_style("ticks")

In [ ]:
mds = pd.read_parquet("0.parquet")

# Throughout layers

In [ ]:
feature = "sentence_RC_attached"

In [ ]:
g = sns.relplot(
    shuffle(mds),
    x="coord_1",
    y="coord_2",
    hue=feature,
    col="representations.layer",
    col_wrap=4,
    kind="scatter",
    height=6,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    linewidth=0.25,
    s=30,
)
g.fig.subplots_adjust(wspace=0.05, hspace=0.125)
g.set_titles(col_template="Layer {col_name}")
g.set_axis_labels("", "")
for ax in g.axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
sns.move_legend(g, loc="lower center", bbox_to_anchor=(0.475, 1), ncol=3, markerscale=2)
plt.savefig(f"../../../paper/figs/BERT_RC_CLS/MDS/{feature}_mds.png", bbox_inches="tight")
plt.savefig(f"../../../paper/figs/BERT_RC_CLS/MDS/{feature}_mds.pdf", bbox_inches="tight")
plt.show()

# Hierarchical clustering

In [ ]:
feature_1 = "subj_NUM"
feature_2 = "sentence_RC_attached"
feature_3 = "verb_ZIPF"
size_1 = max(len(feature_1), mds[feature_1].str.len().max())
hue = feature_1.ljust(size_1) + "    " + feature_2
mds[hue] = mds[feature_1].str.ljust(size_1 + 8) + "    " + mds[feature_2].astype(str)
hue_order = sorted(mds[hue].unique())
tmp = shuffle(mds[mds["representations.layer"] == 7])

In [ ]:
fig = plt.figure(figsize=(10, 10))
values = tmp[feature_3].unique()
params = dict(
    x="coord_1",
    y="coord_2",
    style=feature_3,
    hue=hue,
    hue_order=hue_order,
    palette="tab20",
    markers={values[0]: "X", values[1]: "o"},
    s=30,
    linewidth=0.3,
)
ax = sns.scatterplot(data=tmp[tmp[feature_3] == values[0]], edgecolor="grey", **params)
ax = sns.scatterplot(
    data=tmp[tmp[feature_3] == values[1]], edgecolor="white", ax=ax, **params
)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_xlabel("")
ax.set_ylabel("")

handles, labels = ax.get_legend_handles_labels()
n_hue = len(hue_order)
hue_handles = [Patch(facecolor=h.get_markerfacecolor()) for h in handles[1 : 1 + n_hue]]
handles = handles[0:1] + hue_handles + handles[1 + n_hue : 1 + n_hue + 2] + handles[-1:]
labels = labels[0:1] + hue_order + labels[1 + n_hue : 1 + n_hue + 2] + labels[-1:]
# Make the first label bold
labels[0] = r"$\bf{" + labels[0].replace("_", r"\_").replace("    ", r"\_") + "}$"
labels[-3] = r"$\bf{" + labels[-3].replace("_", r"\_") + "}$"
leg = ax.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    markerscale=2.0,
)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds.pdf", bbox_inches="tight"
)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds.png", bbox_inches="tight"
)
plt.show()

## Walkthrough

In [ ]:
tmp = shuffle(mds[mds["representations.layer"] == 7])
tmp["hue"] = tmp[feature_1]
hue_order = tmp[feature_1].unique().tolist()
tmp["Level"] = 1
tmps = [tmp.copy()]
tmp["hue"] = tmp[feature_2]
hue_order += tmp[feature_2].unique().tolist()
tmp.loc[tmp[feature_1] == tmp[feature_1].iloc[0], "hue"] = "NA"
tmp["Level"] = 2
tmps.append(tmp.copy())
tmp["hue"] = tmp[feature_3]
hue_order += tmp[feature_3].unique().tolist()
tmp.loc[
    (tmp[feature_1] == tmp[feature_1].iloc[0])
    + (tmp[feature_2] == tmp[feature_2].iloc[0]),
    "hue",
] = "NA"
tmp["Level"] = 3
tmps.append(tmp.copy())
tmps = pd.concat(tmps, ignore_index=True)

In [ ]:
palette = sns.color_palette("tab10")
palette = palette[: len(hue_order)] + [sns.color_palette("tab20")[15]]
hue_order += ["NA"]

In [ ]:
g = sns.relplot(
    shuffle(tmps),
    kind="scatter",
    x="coord_1",
    y="coord_2",
    hue="hue",
    hue_order=hue_order,
    col="Level",
    s=15,
    height=6,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    linewidth=0.25,
    palette=palette,
    legend=False,
)
g.set_axis_labels("", "")
g.set_titles(col_template="")
g.fig.subplots_adjust(wspace=0.05)
for ax in g.axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color, label=label)
    for color, label in zip(palette, hue_order)
    if label != "NA"
]
legend_handles.insert(
    0, Line2D([0], [0], color="w", label=r"$\bf{" + feature_1.replace("_", r"\_") + "}$")
)
legend_handles.insert(
    3, Line2D([0], [0], color="w", label=r"$\bf{" + feature_2.replace("_", r"\_") + "}$")
)
legend_handles.insert(
    6, Line2D([0], [0], color="w", label=r"$\bf{" + feature_3.replace("_", r"\_") + "}$")
)
plt.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(-0.7, 1),
    ncols=3,
    frameon=False,
)
plt.savefig(
    f"../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds_full.pdf", bbox_inches="tight"
)
plt.savefig(
    f"../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds_full.png", bbox_inches="tight"
)
plt.show()

## Facets

In [ ]:
feature_1 = "subj_NUM"
feature_2 = "sentence_RC_attached"
feature_3 = "verb_ZIPF"

In [ ]:
mds[[feature_1, feature_2, feature_3]] = mds[[feature_1, feature_2, feature_3]].astype(
    str
)

In [ ]:
mds = mds[mds["representations.layer"] == 7]
values = mds[feature_1].unique()
tmps = []
hue = feature_2 + "    " + feature_3
for v in values:
    tmp = mds.copy()
    tmp[hue] = tmp[feature_2] + "    " + tmp[feature_3]
    tmp.loc[tmp[feature_1] != v, hue] = "Hidden"
    tmp[feature_1] = v
    tmps.append(tmp)
tmps = pd.concat(tmps, ignore_index=True)
hue_order = sorted((tmps[feature_2] + "    " + tmps[feature_3]).unique())
palette = sns.color_palette("tab20")[: len(hue_order)]
palette += ["w", "lightgrey"]
hue_order += ["", "Hidden"]

In [ ]:
g = sns.relplot(
    shuffle(tmps),
    kind="scatter",
    x="coord_1",
    y="coord_2",
    hue=hue,
    hue_order=hue_order,
    col=feature_1,
    s=20,
    height=7,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    linewidth=0.25,
    palette=palette,
)
g.set_axis_labels("", "")
# g.set_titles(col_template="{col_name}")
g.fig.subplots_adjust(wspace=0.05)
for ax in g.axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
# legend_handles = [
#     Line2D([0], [0], marker="o", color="w", markerfacecolor=color, label=label)
#     for color, label in zip(palette, hue_order)
#     if label != "NA"
# ]
# legend_handles.insert(
#     0, Line2D([0], [0], color="w", label=r"$\bf{" + feature_1.replace("_", r"\_") + "}$")
# )
# legend_handles.insert(
#     3, Line2D([0], [0], color="w", label=r"$\bf{" + feature_2.replace("_", r"\_") + "}$")
# )
# legend_handles.insert(
#     6, Line2D([0], [0], color="w", label=r"$\bf{" + feature_3.replace("_", r"\_") + "}$")
# )
# plt.legend(
#     handles=legend_handles,
#     loc="lower center",
#     bbox_to_anchor=(-0.7, 1),
#     ncols=3,
#     frameon=False,
# )
sns.move_legend(g, "center right", bbox_to_anchor=(1, 0.6), markerscale=5)
plt.savefig(
    f"../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds_full_v2.pdf",
    bbox_inches="tight",
)
plt.savefig(
    f"../../../paper/figs/BERT_RC_CLS/MDS/hierarchical_mds_full_v2.png",
    bbox_inches="tight",
)
plt.show()